# TUM-VIE Fusion Bench (JAX + Spyx)

Compare `ConvLIFSNN` vs `IMUConditionedVisualSNN` on Tonic `TUMVIE` for fast 6-DoF regression prototyping.

In [ ]:
from pathlib import Path
import jax
import jax.numpy as jnp
import numpy as np
import haiku as hk
import optax
from tonic.datasets import TUMVIE
from tonic import transforms
import spyx.models as fm

SEED = 0
N_BINS = 24
SPATIAL_FACTOR = 0.25
RECORDING = 'mocap-6dof'
data_root = Path('../data').resolve()
sensor = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR), int(sensor[1] * SPATIAL_FACTOR), sensor[2])
tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_BINS),
])
dataset = TUMVIE(save_to=str(data_root), recording=RECORDING, transform=tfm)
print('len(dataset)=', len(dataset))

In [ ]:
d, t = dataset[0]
x = np.concatenate([np.asarray(d['events_left']), np.asarray(d['events_right'])], axis=1).astype(np.float32)
imu = np.asarray(d['imu'], dtype=np.float32)
imu = imu.mean(axis=0) if imu.ndim == 2 and imu.shape[0] > 0 else np.zeros((6,), dtype=np.float32)

def baseline_forward(xb):
    cfg = fm.ConvConfig(T=int(xb.shape[1]), in_channels=int(xb.shape[2]), hidden_channels=(32, 64), output_dim=6, dt=1.0)
    return fm.ConvLIFSNN(cfg)(xb)

def imu_forward(xb, ub):
    cfg = fm.IMUConditionedConfig(
        vision_cfg=fm.ConvConfig(T=int(xb.shape[1]), in_channels=int(xb.shape[2]), hidden_channels=(32, 64), output_dim=64, dt=1.0),
        imu_dim=int(ub.shape[-1]), fused_dim=128, output_dim=6,
    )
    return fm.IMUConditionedVisualSNN(cfg)(xb, ub)

x_batch = jnp.asarray(np.stack([x] * 4, axis=0))
u_batch = jnp.asarray(np.stack([imu] * 4, axis=0))

base = hk.without_apply_rng(hk.transform(lambda xb: baseline_forward(xb)))
p0 = base.init(jax.random.PRNGKey(SEED), x_batch)
y0 = base.apply(p0, x_batch)

fus = hk.without_apply_rng(hk.transform(lambda xb, ub: imu_forward(xb, ub)))
p1 = fus.init(jax.random.PRNGKey(SEED + 1), x_batch, u_batch)
y1 = fus.apply(p1, x_batch, u_batch)

print('baseline out:', y0.shape)
print('fusion out  :', y1.shape)

Next: replace the single-sample smoke test with mini-batch training and report MSE/MAE over train/val/test splits.